<a href="https://colab.research.google.com/github/nr868848-stack/Sales-Performance-Dashboard/blob/main/Sales_Performance_Dashbord.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

print("Sales Dashboard Project Started Successfully!")

Sales Dashboard Project Started Successfully!


In [ ]:
np.random.seed(42)

n = 3000

products = {
    "Technology": {
        "names": ["Laptop", "Phone", "Monitor", "Keyboard", "Mouse"],
        "subcategories": ["Computers", "Phones", "Accessories", "Accessories", "Accessories"]
    },
    "Furniture": {
        "names": ["Office Chair", "Desk", "Bookshelf", "Table", "Cabinet"],
        "subcategories": ["Chairs", "Tables", "Storage", "Tables", "Storage"]
    },
    "Office Supplies": {
        "names": ["Notebook", "Pen Set", "Printer Paper", "Folder", "Stapler"],
        "subcategories": ["Paper", "Writing", "Paper", "Storage", "Accessories"]
    }
}

categories = np.random.choice(
    list(products.keys()), n, p=[0.35, 0.30, 0.35]
)

product_names = []
subcategories = []

for category in categories:
    idx = np.random.randint(len(products[category]["names"]))
    product_names.append(products[category]["names"][idx])
    subcategories.append(products[category]["subcategories"][idx])

order_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2024-01-01", "2025-12-31"),
        size=n
    )
)

sales = np.round(np.random.lognormal(mean=4.2, sigma=0.8, size=n), 2)
quantity = np.random.randint(1, 9, n)
discount = np.random.choice([0, 0.05, 0.10, 0.15, 0.20], n)
profit = np.round(sales * np.random.uniform(0.05, 0.30, n) - sales * discount, 2)

df = pd.DataFrame({
    "OrderID": [f"ORD{i:05d}" for i in range(1, n + 1)],
    "OrderDate": order_dates,
    "CustomerID": [f"CUST{x:04d}" for x in np.random.randint(1, 801, n)],
    "Segment": np.random.choice(
        ["Consumer", "Corporate", "Small Business"], n
    ),
    "Region": np.random.choice(
        ["Central", "East", "North", "South", "West"], n
    ),
    "State": np.random.choice(
        ["Madhya Pradesh", "Maharashtra", "Delhi", "Gujarat",
         "Rajasthan", "Karnataka", "Uttar Pradesh"], n
    ),
    "Category": categories,
    "SubCategory": subcategories,
    "ProductName": product_names,
    "Sales": sales,
    "Quantity": quantity,
    "Discount": discount,
    "Profit": profit
})

df = df.sort_values("OrderDate").reset_index(drop=True)

display(df.head())
print("Dataset shape:", df.shape)

,OrderID,OrderDate,CustomerID,Segment,Region,State,Category,SubCategory,ProductName,Sales,Quantity,Discount,Profit
0,ORD00415,2024-01-01,CUST0611,Corporate,South,Maharashtra,Furniture,Tables,Desk,49.69,3,0.00,8.44
1,ORD01986,2024-01-01,CUST0799,Corporate,North,Uttar Pradesh,Technology,Accessories,Monitor,44.98,2,0.05,9.19
2,ORD00406,2024-01-02,CUST0760,Small Business,West,Delhi,Office Supplies,Paper,Notebook,42.07,6,0.10,5.87
3,ORD00810,2024-01-02,CUST0140,Corporate,East,Maharashtra,Technology,Accessories,Monitor,29.36,2,0.10,3.52
4,ORD02771,2024-01-02,CUST0116,Small Business,West,Karnataka,Technology,Computers,Laptop,143.95,4,0.05,24.53


Dataset shape: (3000, 13)


In [ ]:
print("Missing values:")
display(df.isnull().sum().to_frame("Missing Count"))

print("Duplicate Order IDs:", df["OrderID"].duplicated().sum())
print("Unique customers:", df["CustomerID"].nunique())
print("Date range:", df["OrderDate"].min(), "to", df["OrderDate"].max())

Missing values:


,Missing Count
OrderID,0
OrderDate,0
CustomerID,0
Segment,0
Region,0
State,0
Category,0
SubCategory,0
ProductName,0
Sales,0


Duplicate Order IDs: 0
Unique customers: 778
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00


In [ ]:
import plotly.express as px

monthly = (
    df.groupby("Month", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
)

fig = px.line(
    monthly, x="Month", y="Sales",
    title="Monthly Sales Trend",
    markers=True
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
region_summary = (
    df.groupby("Region", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .sort_values("Sales", ascending=False)
)

display(region_summary.round(2))

fig = px.bar(
    region_summary,
    x="Region", y="Sales",
    color="Profit",
    title="Sales and Profit by Region",
    hover_data=["Profit"],
    color_continuous_scale="Blues"
)
fig.show()

,Region,Sales,Profit
4,West,62011.89,4997.12
2,North,61770.38,4790.05
1,East,55152.06,4292.26
0,Central,52225.92,4392.41
3,South,47990.85,3576.39


In [ ]:
category_summary = (
    df.groupby("Category", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
)

fig = px.pie(
    category_summary,
    names="Category",
    values="Sales",
    title="Sales Share by Product Category",
    hole=0.4
)

fig.show()

In [ ]:
top_products = (
    df.groupby("ProductName", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .nlargest(10, "Sales")
      .sort_values("Sales")
)

fig = px.bar(
    top_products,
    x="Sales",
    y="ProductName",
    orientation="h",
    title="Top Products by Sales",
    hover_data=["Profit"]
)

fig.show()

In [ ]:
import ipywidgets as widgets

region_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Region"].unique().tolist()),
    value="All",
    description="Region:"
)

category_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Category"].unique().tolist()),
    value="All",
    description="Category:"
)

def update_dashboard(region, category):
    filtered = df.copy()

    if region != "All":
        filtered = filtered[filtered["Region"] == region]

    if category != "All":
        filtered = filtered[filtered["Category"] == category]

    if filtered.empty:
        print("No data for this selection.")
        return

    sales = filtered["Sales"].sum()
    profit = filtered["Profit"].sum()
    orders = filtered["OrderID"].nunique()
    margin = profit / sales * 100 if sales else 0

    print(f"Sales: ${sales:,.2f}")
    print(f"Profit: ${profit:,.2f}")
    print(f"Orders: {orders:,}")
    print(f"Profit Margin: {margin:.2f}%")

    monthly_data = (
        filtered.groupby("Month", as_index=False)["Sales"].sum()
    )

    fig = px.line(
        monthly_data, x="Month", y="Sales",
        title="Filtered Monthly Sales",
        markers=True
    )
    fig.show()

interactive_dashboard = widgets.interactive_output(
    update_dashboard,
    {"region": region_filter, "category": category_filter}
)

display(widgets.VBox([region_filter, category_filter]))
display(interactive_dashboard)

Output()

In [ ]:
import os

os.makedirs("sales_dashboard_project", exist_ok=True)

df.to_csv(
    "sales_dashboard_project/Sales_Data.csv",
    index=False
)

monthly.to_csv(
    "sales_dashboard_project/Monthly_Sales_Summary.csv",
    index=False
)

region_summary.to_csv(
    "sales_dashboard_project/Regional_Performance.csv",
    index=False
)

category_summary.to_csv(
    "sales_dashboard_project/Category_Performance.csv",
    index=False
)

top_products.to_csv(
    "sales_dashboard_project/Top_Products.csv",
    index=False
)

print("All report files saved!")

All report files saved!
